# Flight Fare Prediction

**Regression Project 59**

Full pipeline: EDA → preprocessing → baseline → LazyPredict → FLAML → top-3 evaluation.

## 2. Project Overview

Predict airline ticket prices based on flight attributes: airline, source/destination cities,
departure/arrival time, number of stops, and travel class. This dataset comes from Indian
domestic flights and involves significant datetime parsing and categorical feature work.

Flight pricing is a real-world dynamic pricing problem that combines temporal patterns,
route economics, and airline strategy.

## 3. Learning Objectives

1. Parse datetime features from raw flight data
2. Handle high-cardinality categorical features (airlines, routes)
3. Engineer time-based features (departure hour, journey duration)
4. Understand airline pricing factors
5. Compare LazyPredict, FLAML, and tuned gradient boosting
6. Evaluate with R², RMSE, MAE

## 4. Problem Statement

> Given flight attributes (airline, route, stops, date, duration), predict the **ticket price**.

This is a **regression** task. Primary metric: **R²**; secondary: RMSE, MAE.

## 5. Why This Project Matters

| Stakeholder | Impact |
|---|---|
| Travellers | Find best-value flights |
| Travel agencies | Price comparison tools |
| Airlines | Competitive pricing analysis |
| Revenue management | Dynamic pricing models |
| ML learners | Messy real-world data with datetime parsing |

## 6. Dataset Overview

| Property | Value |
|---|---|
| Source | Kaggle: `nikhilmittal/flight-fare-prediction-mh` |
| Records | ~10,683 |
| Features | Airline, Date_of_Journey, Source, Destination, Route, Dep_Time, Arrival_Time, Duration, Total_Stops, Additional_Info |
| Target | `Price` (INR) |
| Key insight | Airline, stops, and duration are strongest predictors |

## 7. Dataset Source and License Notes

- **Kaggle**: [nikhilmittal/flight-fare-prediction-mh](https://www.kaggle.com/datasets/nikhilmittal/flight-fare-prediction-mh)
- Indian domestic flight data
- License: check Kaggle dataset page
- Downloaded via `kagglehub` at runtime

## 8. Environment Setup

In [ ]:
import subprocess, sys, importlib
for pkg in ['lazypredict', 'flaml', 'kagglehub', 'xgboost', 'lightgbm', 'openpyxl']:
    try: importlib.import_module(pkg)
    except ImportError: subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print('Environment ready.')

## 9. Imports

In [ ]:
import os, warnings, glob, re
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from lazypredict.Supervised import LazyRegressor
from flaml import AutoML
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
SEED = 42
print('All imports loaded.')

## 10. Configuration / Constants

In [ ]:
KAGGLE_SLUG = 'nikhilmittal/flight-fare-prediction-mh'
TARGET = 'Price'
TEST_SIZE = 0.15
VAL_SIZE = 0.15
FLAML_BUDGET = 120
MAX_ROWS = 50000

## 11. Dataset Download or Loading

Download the flight fare dataset. It may be in Excel format.

In [ ]:
import kagglehub
try:
    path = kagglehub.dataset_download(KAGGLE_SLUG)
    print(f'Downloaded to: {path}')
except Exception as e:
    raise RuntimeError(
        f'Dataset download failed: {e}\n'
        'Ensure KAGGLE_API_TOKEN is set. '
        'Fallback: download from https://www.kaggle.com/datasets/nikhilmittal/flight-fare-prediction-mh'
    )
all_files = glob.glob(os.path.join(path, '**', '*.*'), recursive=True)
csv_files = [f for f in all_files if f.endswith('.csv')]
xlsx_files = [f for f in all_files if f.endswith('.xlsx')]
# Prefer train file
train_csv = [f for f in csv_files if 'train' in os.path.basename(f).lower()]
train_xlsx = [f for f in xlsx_files if 'train' in os.path.basename(f).lower()]
if train_csv:
    df_raw = pd.read_csv(train_csv[0])
elif train_xlsx:
    df_raw = pd.read_excel(train_xlsx[0])
elif csv_files:
    df_raw = pd.read_csv(sorted(csv_files, key=os.path.getsize, reverse=True)[0])
elif xlsx_files:
    df_raw = pd.read_excel(sorted(xlsx_files, key=os.path.getsize, reverse=True)[0])
else:
    raise FileNotFoundError(f'No data files in {path}')
print(f'Shape: {df_raw.shape}')
df_raw.head()

## 12. Data Validation Checks

Parse datetime fields, handle missing values, and validate target.

In [ ]:
df = df_raw.copy()
assert TARGET in df.columns, f"Target '{TARGET}' not found in {list(df.columns)}"
print(f'Missing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}\n')
print(f'Duplicated rows: {df.duplicated().sum()}')
df = df.dropna(subset=[TARGET]).drop_duplicates().reset_index(drop=True)
if len(df) > MAX_ROWS:
    df = df.sample(MAX_ROWS, random_state=SEED).reset_index(drop=True)
print(f'Shape: {df.shape}')
print(f'Price range: {df[TARGET].min():,.0f} to {df[TARGET].max():,.0f} INR')

## 13. Exploratory Data Analysis

Flight data has many categorical and datetime features that need careful handling.

In [ ]:
df.describe()

In [ ]:
cat_feats = df.select_dtypes(include=['object']).columns.tolist()
for col in cat_feats[:6]:
    print(f'{col}: {df[col].nunique()} unique')

In [ ]:
# Price by airline
if 'Airline' in df.columns:
    fig, ax = plt.subplots(figsize=(12, 5))
    df.groupby('Airline')[TARGET].mean().sort_values(ascending=False).plot.barh(ax=ax, color='steelblue')
    ax.set_title(f'Mean {TARGET} by Airline')
    plt.tight_layout(); plt.show()

In [ ]:
# Price by number of stops
if 'Total_Stops' in df.columns:
    fig, ax = plt.subplots(figsize=(8, 4))
    df.groupby('Total_Stops')[TARGET].mean().sort_values().plot.bar(ax=ax, color='coral')
    ax.set_title(f'Mean {TARGET} by Total Stops')
    plt.tight_layout(); plt.show()

## 14. Target Analysis

Flight prices are heavily right-skewed with budget fares dominating.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df[TARGET], bins=50, color='coral', alpha=0.7)
axes[0].set_title(f'{TARGET} Distribution (INR)')
axes[1].hist(np.log1p(df[TARGET]), bins=50, color='seagreen', alpha=0.7)
axes[1].set_title(f'log({TARGET}) Distribution')
plt.tight_layout(); plt.show()
print(f'Mean: {df[TARGET].mean():,.0f} INR')
print(f'Median: {df[TARGET].median():,.0f} INR')
print(f'Skewness: {df[TARGET].skew():.2f}')

## 15. Train / Validation / Test Split

70/15/15 split. We first extract numeric features from datetime/text columns.

In [ ]:
# Parse datetime features
def parse_flight_features(d):
    d = d.copy()
    # Parse Date_of_Journey
    if 'Date_of_Journey' in d.columns:
        d['journey_day'] = pd.to_datetime(d['Date_of_Journey'], dayfirst=True, errors='coerce').dt.day
        d['journey_month'] = pd.to_datetime(d['Date_of_Journey'], dayfirst=True, errors='coerce').dt.month
        d = d.drop(columns=['Date_of_Journey'])
    # Parse Dep_Time
    if 'Dep_Time' in d.columns:
        dep = pd.to_datetime(d['Dep_Time'], errors='coerce')
        d['dep_hour'] = dep.dt.hour
        d['dep_min'] = dep.dt.minute
        d = d.drop(columns=['Dep_Time'])
    # Parse Arrival_Time
    if 'Arrival_Time' in d.columns:
        d = d.drop(columns=['Arrival_Time'])
    # Parse Duration
    if 'Duration' in d.columns:
        def parse_dur(x):
            if pd.isna(x): return np.nan
            h = re.findall(r'(\d+)h', str(x))
            m = re.findall(r'(\d+)m', str(x))
            return int(h[0])*60 + int(m[0]) if h and m else (int(h[0])*60 if h else (int(m[0]) if m else np.nan))
        d['duration_min'] = d['Duration'].apply(parse_dur)
        d = d.drop(columns=['Duration'])
    # Parse Total_Stops
    if 'Total_Stops' in d.columns:
        stops_map = {'non-stop': 0, '1 stop': 1, '2 stops': 2, '3 stops': 3, '4 stops': 4}
        d['num_stops'] = d['Total_Stops'].map(stops_map).fillna(0).astype(int)
        d = d.drop(columns=['Total_Stops'])
    # Drop Route and Additional_Info (high cardinality / low value)
    for col in ['Route', 'Additional_Info']:
        if col in d.columns: d = d.drop(columns=[col])
    return d

df = parse_flight_features(df)
X = df.drop(columns=[TARGET])
y = df[TARGET]
X_tmp, X_test, y_tmp, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(X_tmp, y_tmp, test_size=VAL_SIZE/(1-TEST_SIZE), random_state=SEED)
print(f'Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}')

## 16. Preprocessing Strategy

- **Numeric**: duration, stops, departure time → impute + scale
- **Categorical**: Airline, Source, Destination → OneHotEncoder

In [ ]:
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
transformers = []
if num_cols:
    transformers.append(('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), num_cols))
if cat_cols:
    transformers.append(('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat_cols))
preprocessor = ColumnTransformer(transformers=transformers, remainder='drop')
print(f'Numeric: {len(num_cols)}, Categorical: {len(cat_cols)}')

## 17. Feature Engineering

- **is_direct**: Whether the flight is non-stop
- **time_of_day**: Morning/afternoon/evening/night from departure hour
- Features already extracted in parsing step (duration_min, dep_hour, num_stops)

In [ ]:
def engineer_features(d):
    d = d.copy()
    if 'num_stops' in d.columns:
        d['is_direct'] = (d['num_stops'] == 0).astype(int)
    if 'dep_hour' in d.columns:
        d['time_of_day'] = pd.cut(d['dep_hour'].fillna(12), bins=[-1,6,12,18,24],
                                  labels=['night','morning','afternoon','evening'])
    return d

X_train = engineer_features(X_train)
X_val = engineer_features(X_val)
X_test = engineer_features(X_test)
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
transformers = []
if num_cols:
    transformers.append(('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), num_cols))
if cat_cols:
    transformers.append(('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat_cols))
preprocessor = ColumnTransformer(transformers=transformers, remainder='drop')
print(f'Features: {X_train.shape[1]}')

## 18. Baseline Model

In [ ]:
results = {}
for name, reg in [
    ('Dummy (mean)', DummyRegressor(strategy='mean')),
    ('LinearRegression', LinearRegression()),
    ('RandomForest', RandomForestRegressor(n_estimators=200, random_state=SEED, n_jobs=-1))
]:
    pipe = Pipeline([('pre', preprocessor), ('reg', reg)])
    pipe.fit(X_train, y_train)
    yp = pipe.predict(X_val)
    r = {'RMSE': np.sqrt(mean_squared_error(y_val, yp)),
         'MAE': mean_absolute_error(y_val, yp),
         'R2': r2_score(y_val, yp)}
    results[name] = r
    print(f"{name}: R2={r['R2']:.4f}, RMSE={r['RMSE']:,.0f} INR, MAE={r['MAE']:,.0f} INR")

## 19. LazyPredict Benchmark

In [ ]:
X_train_lp = preprocessor.fit_transform(X_train)
X_val_lp = preprocessor.transform(X_val)
lazy = LazyRegressor(verbose=0, ignore_warnings=True, random_state=SEED)
models_lp, _ = lazy.fit(X_train_lp, X_val_lp, y_train, y_val)
models_lp.head(15)

## 20. FLAML AutoML Run

In [ ]:
automl = AutoML()
automl.fit(X_train_lp, y_train, task='regression', metric='r2',
           time_budget=FLAML_BUDGET, seed=SEED, verbose=0)
print(f'Best estimator: {automl.best_estimator}')
yp_fl = automl.predict(X_val_lp)
results['FLAML'] = {'RMSE': np.sqrt(mean_squared_error(y_val, yp_fl)),
                    'MAE': mean_absolute_error(y_val, yp_fl),
                    'R2': r2_score(y_val, yp_fl)}
print(f"FLAML: R2={results['FLAML']['R2']:.4f}, RMSE={results['FLAML']['RMSE']:,.0f}")

## 21. Top 3 Model Selection

In [ ]:
try:
    from lightgbm import LGBMRegressor; has_lgbm = True
except ImportError: has_lgbm = False
try:
    from xgboost import XGBRegressor; has_xgb = True
except ImportError: has_xgb = False
top3 = {}
if has_lgbm:
    top3['LightGBM'] = LGBMRegressor(n_estimators=500, learning_rate=0.05, max_depth=7, random_state=SEED, verbose=-1, n_jobs=-1)
else:
    from sklearn.ensemble import ExtraTreesRegressor
    top3['ExtraTrees'] = ExtraTreesRegressor(n_estimators=500, random_state=SEED, n_jobs=-1)
if has_xgb:
    top3['XGBoost'] = XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=7, random_state=SEED, verbosity=0, n_jobs=-1)
else:
    from sklearn.ensemble import AdaBoostRegressor
    top3['AdaBoost'] = AdaBoostRegressor(n_estimators=300, random_state=SEED)
top3['GradBoosting'] = GradientBoostingRegressor(n_estimators=400, learning_rate=0.05, max_depth=6, random_state=SEED)
print(f'Top 3: {list(top3.keys())}')

## 22. Final Training and Evaluation of Top 3

In [ ]:
X_tr_t = preprocessor.fit_transform(X_train)
X_te_t = preprocessor.transform(X_test)
final = {}
predictions = {}
for name, mdl in top3.items():
    mdl.fit(X_tr_t, y_train)
    yp = mdl.predict(X_te_t)
    predictions[name] = yp
    final[name] = {'RMSE': np.sqrt(mean_squared_error(y_test, yp)),
                   'MAE': mean_absolute_error(y_test, yp),
                   'R2': r2_score(y_test, yp)}
    print(f"{name}: R2={final[name]['R2']:.4f}, RMSE={final[name]['RMSE']:,.0f}, MAE={final[name]['MAE']:,.0f}")
pd.DataFrame(final).T.sort_values('R2', ascending=False)

In [ ]:
fig, axes = plt.subplots(1, len(top3), figsize=(6*len(top3), 5))
if len(top3) == 1: axes = [axes]
for ax, (name, yp) in zip(axes, predictions.items()):
    ax.scatter(y_test, yp, alpha=0.3, s=10)
    mn, mx = min(y_test.min(), yp.min()), max(y_test.max(), yp.max())
    ax.plot([mn, mx], [mn, mx], 'r--', lw=2)
    ax.set_xlabel('Actual (INR)'); ax.set_ylabel('Predicted (INR)')
    ax.set_title(f"{name} (R2={final[name]['R2']:.3f})")
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(top3), figsize=(6*len(top3), 5))
if len(top3) == 1: axes = [axes]
for ax, (name, yp) in zip(axes, predictions.items()):
    residuals = y_test.values - yp
    ax.scatter(yp, residuals, alpha=0.3, s=10)
    ax.axhline(0, color='red', linestyle='--')
    ax.set_xlabel('Predicted'); ax.set_ylabel('Residual')
    ax.set_title(f'{name} Residuals')
plt.tight_layout(); plt.show()

## 23. Error Analysis

In [ ]:
best_name = max(final, key=lambda k: final[k]['R2'])
best_preds = predictions[best_name]
abs_errors = np.abs(y_test.values - best_preds)
print(f'Best model: {best_name}')
print(f'Mean absolute error: {abs_errors.mean():,.0f} INR')
print(f'Median absolute error: {np.median(abs_errors):,.0f} INR')
print(f'90th percentile error: {np.percentile(abs_errors, 90):,.0f} INR')

## 24. Interpretation and Business Insight

- **Airline** is a major price driver — premium carriers (Jet Airways Business) charge 5–10x more
- **Number of stops** strongly affects price — direct flights can be cheaper or premium
- **Duration** correlates with distance and price
- **Departure time** has mild effects — early morning and late night are cheaper
- **Seasonality** (journey month) captures peak travel period pricing
- The model captures airline-route-time interactions that drive dynamic pricing

## 25. Limitations

1. Indian domestic flights only — not generalizable internationally
2. No booking lead time (how far in advance)
3. No seat class detail beyond airline
4. Static snapshot — prices change dynamically
5. No competitor pricing or demand data

## 26. How to Improve This Project

1. Add booking lead time as a feature
2. Include day-of-week for weekday/weekend pricing
3. Log-transform price for better residual behavior
4. Target encode high-cardinality route features
5. Time-series approach for dynamic pricing

## 27. Production Considerations

- Real-time fare scraping and prediction
- Price alert system for travellers
- Integration with booking platforms
- Regular retraining as pricing patterns change
- Currency and inflation adjustment

## 28. Common Mistakes

1. Not parsing Duration from text (e.g., '2h 50m') to minutes
2. Treating Total_Stops as numeric without mapping
3. Keeping Route as raw text with 100+ unique values
4. Not handling Date_of_Journey — extract day and month
5. Ignoring extreme outlier prices

## 29. Mini Challenge / Exercises

1. Try log-transforming Price and compare R²
2. Add day-of-week feature from Date_of_Journey
3. Build separate models for budget vs premium airlines
4. Extract intermediate stops from Route column
5. Compare OHE vs target encoding for Airline

## 30. Final Summary / Key Takeaways

| Aspect | Detail |
|---|---|
| Task | Regression — predict flight ticket price |
| Dataset | Flight Fare (~10K records) |
| Difficulty | Medium |
| Key insight | Airline brand and stops dominate pricing |
| Best approach | Gradient boosting with parsed datetime features |
| Primary metric | R² |
| Baseline comparison | Feature engineering is critical for performance |